# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
from collections import Counter
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg
from nltk.tokenize import RegexpTokenizer
from scipy.sparse import issparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)

root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
ruta_nltk = str(root_dir / "data" / "nltk_data")
if ruta_nltk not in nltk.data.path:
    nltk.data.path.insert(0, ruta_nltk)

tokenizador = RegexpTokenizer(r"[A-Za-z]+(?:['’][A-Za-z]+)?")


def tokenizar(texto):
    return tokenizador.tokenize(texto.lower())


documentos = sorted(str(doc) for doc in gutenberg.fileids())
textos = [gutenberg.raw(doc) for doc in documentos]
corpus = pd.DataFrame({
    "doc_id": documentos,
    "Título": [texto.splitlines()[0].strip().strip("[]") for texto in textos],
    "Texto": textos,
})
corpus["tokens"] = corpus["Texto"].map(tokenizar)
conteos = [Counter(tokens) for tokens in corpus["tokens"]]
longitudes = np.array([len(tokens) for tokens in corpus["tokens"]], dtype=float)
N = len(corpus)
avgdl = float(longitudes.mean())
df = Counter()
for conteo in conteos:
    df.update(conteo.keys())
vocabulario = sorted(df)
idf_tfidf = {t: np.log(N / df[t]) + 1 for t in vocabulario}
idf_bm25 = {t: np.log1p((N - df[t] + 0.5) / (df[t] + 0.5)) for t in vocabulario}
estadisticas_terminos = pd.DataFrame({
    "Término": vocabulario,
    "DF": [df[t] for t in vocabulario],
    "IDF TF-IDF": [idf_tfidf[t] for t in vocabulario],
    "IDF BM25": [idf_bm25[t] for t in vocabulario],
}).set_index("Término")

with pd.option_context("display.max_colwidth", 55):
    display(corpus)
print(f"N={N}; vocabulario={len(vocabulario)}; avgdl={avgdl:.2f} tokens")
display(estadisticas_terminos.sort_values(["DF", "Término"]).iloc[np.r_[0:5, -5:0]])


N=18; vocabulario=43495; avgdl=117718.72 tokens


,doc_id,Título,Texto,tokens
0,austen-emma.txt,Emma by Jane Austen 1816,[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAPTER I...,"[emma, by, jane, austen, volume, i, chapter, i, emm..."
1,austen-persuasion.txt,Persuasion by Jane Austen 1818,[Persuasion by Jane Austen 1818]\n\n\nChapter 1\n\n...,"[persuasion, by, jane, austen, chapter, sir, walter..."
2,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,[Sense and Sensibility by Jane Austen 1811]\n\nCHAP...,"[sense, and, sensibility, by, jane, austen, chapter..."
3,bible-kjv.txt,The King James Bible,[The King James Bible]\n\nThe Old Testament of the ...,"[the, king, james, bible, the, old, testament, of, ..."
4,blake-poems.txt,Poems by William Blake 1789,[Poems by William Blake 1789]\n\n \nSONGS OF INNOCE...,"[poems, by, william, blake, songs, of, innocence, a..."
5,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,[Stories to Tell to Children by Sara Cone Bryant 19...,"[stories, to, tell, to, children, by, sara, cone, b..."
6,burgess-busterbrown.txt,The Adventures of Buster Bear by Thornton W. Burges...,[The Adventures of Buster Bear by Thornton W. Burge...,"[the, adventures, of, buster, bear, by, thornton, w..."
7,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carroll 1865,[Alice's Adventures in Wonderland by Lewis Carroll ...,"[alice's, adventures, in, wonderland, by, lewis, ca..."
8,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,[The Ball and The Cross by G.K. Chesterton 1909]\n\...,"[the, ball, and, the, cross, by, g, k, chesterton, ..."
9,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton 1914,[The Wisdom of Father Brown by G. K. Chesterton 191...,"[the, wisdom, of, father, brown, by, g, k, chestert..."


,DF,IDF TF-IDF,IDF BM25
Término,,,
a'lee,1,3.890372,2.538974
a'low,1,3.890372,2.538974
a'm,1,3.890372,2.538974
a'ready,1,3.890372,2.538974
a'shiver,1,3.890372,2.538974
write,18,1.000000,0.026668
yet,18,1.000000,0.026668
you,18,1.000000,0.026668
young,18,1.000000,0.026668


**Colección y variantes.** Se conservan los 18 documentos NLTK Gutenberg, el texto completo y el preprocesamiento del notebook 03: minúsculas, letras ASCII y apóstrofos internos, sin eliminar palabras vacías ni aplicar stemming o lematización. La tabla `corpus` conserva el texto original y la lista completa de tokens; su visualización abrevia los campos largos.

Para TF-IDF se mantiene la variante anterior: TF cruda e $IDF(t)=1+\ln(N/df(t))$, con normalización L2 de documentos y consultas. El ejemplo de la página 15 de `bm25.md` emplea IDF suavizada; aquí se declara la variante anterior para conservar la comparación entre prácticas.

Para BM25 se usa la IDF positiva de las páginas 5–6 de `bm25.md`:

$$IDF_{BM25}(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right),\qquad BM25(q,d)=\sum_{t\in\operatorname{únicos}(q)}IDF_{BM25}(t)\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/avgdl)}.$$

`estadisticas_terminos` contiene DF y ambas IDF para todo el vocabulario. Las longitudes cuentan tokens procesados.

**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
def mismos_tokens(tokens):
    return tokens


vectorizador = TfidfVectorizer(
    analyzer=mismos_tokens, vocabulary={t: j for j, t in enumerate(vocabulario)},
    smooth_idf=False, sublinear_tf=False, norm="l2", dtype=np.float64,
)
matriz_tfidf = vectorizador.fit_transform(corpus["tokens"]).tocsr()
assert issparse(matriz_tfidf)
np.testing.assert_allclose(vectorizador.idf_, [idf_tfidf[t] for t in vocabulario])


def aportes_bm25(tokens_consulta, k1=1.2, b=0.75):
    if not np.isfinite(k1) or k1 < 0 or not np.isfinite(b) or not 0 <= b <= 1:
        raise ValueError("Se requiere k1 >= 0 y 0 <= b <= 1, ambos finitos.")
    terminos = sorted(set(tokens_consulta))
    aportes = np.zeros((N, len(terminos)))
    K = k1 * (1 - b + b * longitudes / avgdl)
    for j, termino in enumerate(terminos):
        frecuencias = np.array([c.get(termino, 0) for c in conteos], dtype=float)
        factor = np.divide(
            frecuencias * (k1 + 1), frecuencias + K,
            out=np.zeros(N), where=frecuencias + K > 0,
        )
        aportes[:, j] = idf_bm25.get(termino, 0) * factor
    return pd.DataFrame(aportes, index=documentos, columns=terminos)


def ranking(puntuaciones, limite=10):
    tabla = corpus[["doc_id", "Título"]].copy()
    tabla["Score"] = np.asarray(puntuaciones).ravel()
    tabla = tabla.loc[tabla["Score"] > 0].sort_values(
        ["Score", "doc_id"], ascending=[False, True],
    ).head(limite).reset_index(drop=True)
    tabla.index = pd.RangeIndex(1, len(tabla) + 1, name="Posición")
    return tabla


consultas = pd.DataFrame([
    ("the and", "Explorar documentos que contienen estas dos palabras funcionales muy frecuentes."),
    ("whale", "Encontrar textos centrados en ballenas con una consulta de un solo término."),
    ("whale sea ship ocean", "Encontrar relatos de ballenas y navegación marítima."),
    ("love marriage family", "Encontrar historias de amor, matrimonio y relaciones familiares."),
    ("god heaven earth", "Encontrar textos religiosos sobre Dios, el cielo y la creación."),
    ("murder king ghost", "Encontrar tragedias con asesinatos, reyes y apariciones."),
    ("queequeg harpoon leviathan", "Encontrar pasajes sobre Queequeg y la caza de ballenas con términos menos frecuentes."),
    ("love marriage family father mother home", "Encontrar relatos familiares que también mencionen padres y hogar."),
], columns=["Consulta", "Necesidad del usuario"])
consultas["tokens"] = consultas["Consulta"].map(tokenizar)
consultas["Longitud"] = consultas["tokens"].map(len)
consultas["DF por término"] = consultas["tokens"].map(
    lambda tokens: "; ".join(f"{t}: {df.get(t, 0)}/{N}" for t in tokens)
)
with pd.option_context("display.max_colwidth", None):
    display(consultas.drop(columns="tokens"))

idf_antes = vectorizador.idf_.copy()
df_antes = df.copy()
longitudes_antes = longitudes.copy()
consultas_tfidf = vectorizador.transform(consultas["tokens"])
puntuaciones_tfidf = cosine_similarity(consultas_tfidf, matriz_tfidf)
puntuaciones_bm25 = np.vstack([
    aportes_bm25(tokens).sum(axis=1).to_numpy() for tokens in consultas["tokens"]
])
rankings_tfidf = [ranking(scores) for scores in puntuaciones_tfidf]
rankings_bm25 = [ranking(scores) for scores in puntuaciones_bm25]
np.testing.assert_array_equal(vectorizador.idf_, idf_antes)
assert df == df_antes
np.testing.assert_array_equal(longitudes, longitudes_antes)
assert np.isfinite(puntuaciones_bm25).all()

for texto in ["", "123 !!!", "zzqxvnotincorpus"]:
    tokens = tokenizar(texto)
    assert not any(t in df for t in tokens)
    assert ranking(cosine_similarity(vectorizador.transform([tokens]), matriz_tfidf)[0]).empty
    assert ranking(aportes_bm25(tokens).sum(axis=1)).empty
np.testing.assert_allclose(
    aportes_bm25(["whale", "whale"]), aportes_bm25(["whale"]),
)
print("Matriz TF-IDF dispersa:", matriz_tfidf.shape, "con", matriz_tfidf.nnz, "pesos no nulos")
print("Consultas vacías y fuera del vocabulario: dos rankings vacíos; estadísticas conservadas.")


Matriz TF-IDF dispersa: (18, 43495) con 123672 pesos no nulos
Consultas vacías y fuera del vocabulario: dos rankings vacíos; estadísticas conservadas.


,Consulta,Necesidad del usuario,Longitud,DF por término
0,the and,Explorar documentos que contienen estas dos palabras funcionales muy frecuentes.,2,the: 18/18; and: 18/18
1,whale,Encontrar textos centrados en ballenas con una consulta de un solo término.,1,whale: 6/18
2,whale sea ship ocean,Encontrar relatos de ballenas y navegación marítima.,4,whale: 6/18; sea: 16/18; ship: 13/18; ocean: 9/18
3,love marriage family,"Encontrar historias de amor, matrimonio y relaciones familiares.",3,love: 15/18; marriage: 12/18; family: 13/18
4,god heaven earth,"Encontrar textos religiosos sobre Dios, el cielo y la creación.",3,god: 16/18; heaven: 13/18; earth: 17/18
5,murder king ghost,"Encontrar tragedias con asesinatos, reyes y apariciones.",3,murder: 9/18; king: 15/18; ghost: 9/18
6,queequeg harpoon leviathan,Encontrar pasajes sobre Queequeg y la caza de ballenas con términos menos frecuentes.,3,queequeg: 1/18; harpoon: 2/18; leviathan: 3/18
7,love marriage family father mother home,Encontrar relatos familiares que también mencionen padres y hogar.,6,love: 15/18; marriage: 12/18; family: 13/18; father: 16/18; mother: 17/18; home: 18/18


**Elección de consultas.** La tabla muestra la necesidad, longitud y DF de cada término sobre el mismo N. `the and` sirve como control de términos muy frecuentes; la consulta de Queequeg introduce términos menos frecuentes. Se incluyen una consulta de un término y otra de seis. Son necesidades propuestas para el experimento, sin juicios de relevancia asociados.

El vectorizador se ajusta una sola vez; las consultas solo usan `transform`. La matriz documento-término sigue en CSR. Solo se materializa la pequeña tabla de ocho consultas por 18 puntuaciones. BM25 suma una contribución por término distinto y conserva N, DF, IDF y avgdl en todas las consultas. Los rankings filtran scores positivos antes de limitar a diez, y los empates exactos se resuelven por el identificador como cadena.

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [3]:
resumen = []
for i, consulta in consultas.iterrows():
    rt, rb = rankings_tfidf[i], rankings_bm25[i]
    tabla = pd.DataFrame({
        "ID y título TF-IDF": rt["doc_id"] + " | " + rt["Título"],
        "Coseno": rt["Score"],
        "ID y título BM25": rb["doc_id"] + " | " + rb["Título"],
        "Score BM25": rb["Score"],
    }).rename_axis("Posición")
    print(f"Consulta {i + 1}: {consulta['Consulta']}")
    with pd.option_context("display.max_colwidth", None):
        display(tabla.round(8).fillna("—"))
    resumen.append({
        "Consulta": consulta["Consulta"],
        "Tamaño TF-IDF": len(rt), "Tamaño BM25": len(rb),
        "IDs comunes top-10": len(set(rt["doc_id"]) & set(rb["doc_id"])),
        "Primero TF-IDF": rt.iloc[0]["doc_id"] if len(rt) else None,
        "Primero BM25": rb.iloc[0]["doc_id"] if len(rb) else None,
    })
display(pd.DataFrame(resumen))

consulta_elegida = "love marriage family"
tokens_elegidos = tokenizar(consulta_elegida)
i_consulta = consultas.index[consultas["Consulta"] == consulta_elegida][0]
docs_elegidos = ["blake-poems.txt", "austen-sense.txt"]
aportes_base = aportes_bm25(tokens_elegidos)
filas = []
for doc in docs_elegidos:
    i = documentos.index(doc)
    for termino in sorted(set(tokens_elegidos)):
        filas.append({
            "doc_id": doc, "|d|": int(longitudes[i]), "|d|/avgdl": longitudes[i] / avgdl,
            "Término": termino, "f(t,d)": conteos[i].get(termino, 0),
            "df(t)": df[termino], "IDF BM25": idf_bm25[termino],
            "Aporte BM25": aportes_base.loc[doc, termino],
        })
detalle_documentos = pd.DataFrame(filas)
print("Consulta analizada:", consulta_elegida)
display(detalle_documentos.round(8))
comparacion_documentos = []
for doc in docs_elegidos:
    i = documentos.index(doc)
    rt = ranking(puntuaciones_tfidf[i_consulta], limite=N)
    rb = ranking(puntuaciones_bm25[i_consulta], limite=N)
    comparacion_documentos.append({
        "doc_id": doc,
        "Posición TF-IDF": int(rt.index[rt["doc_id"] == doc][0]),
        "Coseno": puntuaciones_tfidf[i_consulta, i],
        "Posición BM25": int(rb.index[rb["doc_id"] == doc][0]),
        "Score BM25": puntuaciones_bm25[i_consulta, i],
    })
display(pd.DataFrame(comparacion_documentos).round(8))
np.testing.assert_allclose(aportes_base.sum(axis=1), puntuaciones_bm25[i_consulta])


Consulta 1: the and
Consulta 2: whale
Consulta 3: whale sea ship ocean
Consulta 4: love marriage family
Consulta 5: god heaven earth
Consulta 6: murder king ghost
Consulta 7: queequeg harpoon leviathan
Consulta 8: love marriage family father mother home
Consulta analizada: love marriage family


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,bible-kjv.txt | The King James Bible,0.792296,bible-kjv.txt | The King James Bible,0.117327
2,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.787994,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.117319
3,blake-poems.txt | Poems by William Blake 1789,0.780186,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,0.117315
4,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,0.760094,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,0.117311
5,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,0.691279,"edgeworth-parents.txt | The Parent's Assistant, by Maria Edgeworth",0.117310
6,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,0.666741,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,0.117309
7,milton-paradise.txt | Paradise Lost by John Milton 1667,0.666720,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,0.117307
8,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,0.638347,milton-paradise.txt | Paradise Lost by John Milton 1667,0.117306
9,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,0.611695,austen-emma.txt | Emma by Jane Austen 1816,0.117305


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,0.108281,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,2.355626
2,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,0.006502,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,2.267115
3,shakespeare-hamlet.txt | The Tragedie of Hamlet by William Shakespeare 1599,0.001540,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,1.888399
4,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.000757,shakespeare-hamlet.txt | The Tragedie of Hamlet by William Shakespeare 1599,1.867655
5,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,0.000495,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,1.611758
6,bible-kjv.txt | The King James Bible,0.000041,bible-kjv.txt | The King James Bible,0.565885


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,0.093609,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,4.899939
2,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.011061,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,4.543125
3,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,0.006989,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,4.408703
4,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,0.005616,shakespeare-hamlet.txt | The Tragedie of Hamlet by William Shakespeare 1599,3.746641
5,milton-paradise.txt | Paradise Lost by John Milton 1667,0.005551,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,3.692610
6,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,0.004704,milton-paradise.txt | Paradise Lost by John Milton 1667,2.141403
7,shakespeare-hamlet.txt | The Tragedie of Hamlet by William Shakespeare 1599,0.002596,shakespeare-macbeth.txt | The Tragedie of Macbeth by William Shakespeare 1603,1.856798
8,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,0.002382,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,1.844821
9,bible-kjv.txt | The King James Bible,0.001913,bible-kjv.txt | The King James Bible,1.561628


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,blake-poems.txt | Poems by William Blake 1789,0.026316,austen-sense.txt | Sense and Sensibility by Jane Austen 1811,2.078310
2,austen-persuasion.txt | Persuasion by Jane Austen 1818,0.013939,austen-persuasion.txt | Persuasion by Jane Austen 1818,2.071438
3,austen-sense.txt | Sense and Sensibility by Jane Austen 1811,0.013203,austen-emma.txt | Emma by Jane Austen 1816,2.061886
4,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.012894,bible-kjv.txt | The King James Bible,1.844618
5,austen-emma.txt | Emma by Jane Austen 1816,0.010720,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,1.844369
6,milton-paradise.txt | Paradise Lost by John Milton 1667,0.009557,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,1.840420
7,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,0.005536,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,1.670943
8,"edgeworth-parents.txt | The Parent's Assistant, by Maria Edgeworth",0.005333,milton-paradise.txt | Paradise Lost by John Milton 1667,1.543360
9,bible-kjv.txt | The King James Bible,0.002937,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,1.519093


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,milton-paradise.txt | Paradise Lost by John Milton 1667,0.086136,milton-paradise.txt | Paradise Lost by John Milton 1667,1.239331
2,bible-kjv.txt | The King James Bible,0.036622,bible-kjv.txt | The King James Bible,1.233446
3,blake-poems.txt | Poems by William Blake 1789,0.024922,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,1.219280
4,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,0.016613,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,1.206204
5,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.013885,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,1.200309
6,shakespeare-hamlet.txt | The Tragedie of Hamlet by William Shakespeare 1599,0.010109,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,1.193964
7,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,0.008027,blake-poems.txt | Poems by William Blake 1789,1.190676
8,shakespeare-macbeth.txt | The Tragedie of Macbeth by William Shakespeare 1603,0.007880,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,1.126402
9,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,0.006895,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,1.096461


... (2 outputs left out)

,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,blake-poems.txt | Poems by William Blake 1789,0.043527,austen-persuasion.txt | Persuasion by Jane Austen 1818,2.612671
2,austen-persuasion.txt | Persuasion by Jane Austen 1818,0.021007,austen-sense.txt | Sense and Sensibility by Jane Austen 1811,2.611251
3,austen-sense.txt | Sense and Sensibility by Jane Austen 1811,0.020384,austen-emma.txt | Emma by Jane Austen 1816,2.603645
4,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,0.018802,bible-kjv.txt | The King James Bible,2.381536
5,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,0.018694,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,2.380169
6,austen-emma.txt | Emma by Jane Austen 1816,0.017511,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,2.320888
7,"edgeworth-parents.txt | The Parent's Assistant, by Maria Edgeworth",0.017115,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,2.216225
8,shakespeare-hamlet.txt | The Tragedie of Hamlet by William Shakespeare 1599,0.015736,milton-paradise.txt | Paradise Lost by John Milton 1667,2.071733
9,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.014576,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,2.018568


,Consulta,Tamaño TF-IDF,Tamaño BM25,IDs comunes top-10,Primero TF-IDF,Primero BM25
0,the and,10,10,7,bible-kjv.txt,bible-kjv.txt
1,whale,6,6,6,melville-moby_dick.txt,melville-moby_dick.txt
2,whale sea ship ocean,10,10,8,melville-moby_dick.txt,melville-moby_dick.txt
3,love marriage family,10,10,8,blake-poems.txt,austen-sense.txt
4,god heaven earth,10,10,8,milton-paradise.txt,milton-paradise.txt
5,murder king ghost,10,10,7,shakespeare-hamlet.txt,shakespeare-hamlet.txt
6,queequeg harpoon leviathan,4,4,4,melville-moby_dick.txt,melville-moby_dick.txt
7,love marriage family father mother home,10,10,7,blake-poems.txt,austen-persuasion.txt


,doc_id,|d|,|d|/avgdl,Término,"f(t,d)",df(t),IDF BM25,Aporte BM25
0,blake-poems.txt,6837,0.058079,family,0,13,0.341749,0.000000
1,blake-poems.txt,6837,0.058079,love,29,15,0.203599,0.442542
2,blake-poems.txt,6837,0.058079,marriage,1,12,0.418710,0.681197
3,austen-sense.txt,119912,1.018632,family,83,13,0.341749,0.740986
4,austen-sense.txt,119912,1.018632,love,77,15,0.203599,0.440950
5,austen-sense.txt,119912,1.018632,marriage,44,12,0.418710,0.896375


,doc_id,Posición TF-IDF,Coseno,Posición BM25,Score BM25
0,blake-poems.txt,1,0.026316,12,1.123739
1,austen-sense.txt,3,0.013203,1,2.078310


**Cómo leer la comparación.** La intersección cuenta IDs compartidos, no posiciones: diez IDs comunes no implican el mismo orden. Las puntuaciones de ambos métodos tienen escalas distintas; el coseno solo explica su propia ordenación y BM25 la suya. Sin juicios de relevancia no se puede afirmar cuál recupera mejor.

**Caso observado: `love marriage family`.** `blake-poems.txt` pasa del puesto 1 de TF-IDF (coseno 0.02631561) al 12 de BM25 (1.12373876); `austen-sense.txt` pasa del 3 (0.01320326) al 1 (2.07830998). Los cosenos muestran que el vector normalizado de Blake se alinea más con esta consulta. Esa normalización considera todos los pesos del documento, no solo su cantidad de tokens.

Blake tiene 6 837 tokens (0.0581 veces la media), frente a 119 912 de Austen (1.0186 veces). Con BM25, las 29 apariciones de `love` en Blake aportan 0.442542, casi lo mismo que las 77 de Austen (0.440950): la saturación y el menor tamaño de Blake reducen la ventaja de repetir más. Pero `family` está ausente en Blake y aporta 0.740986 en Austen. `marriage` tiene menor DF (12) que `family` (13) y `love` (15), por lo que su IDF es mayor; sus 44 apariciones en Austen aportan 0.896375, frente a 0.681197 por una aparición en Blake. La suma de esos aportes explica la ventaja BM25 de Austen, aunque Blake reciba una fuerte compensación por longitud. Estas cifras proceden de las tablas anteriores, calculadas sin redondear internamente.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [4]:
configuraciones = [
    ("Base", 1.2, 0.75),
    ("Sin ajuste por longitud", 1.2, 0.0),
    ("Mayor efecto de las repeticiones", 2.0, 0.75),
]
comparacion_parametros = corpus[["doc_id"]].copy().set_index("doc_id")
comparacion_parametros["|d|"] = longitudes.astype(int)
comparacion_parametros["|d|/avgdl"] = longitudes / avgdl
filas_aportes = []
for nombre, k1, b in configuraciones:
    aportes = aportes_bm25(tokens_elegidos, k1=k1, b=b)
    scores = aportes.sum(axis=1)
    rt = ranking(scores, limite=N)
    posiciones = pd.Series(rt.index.to_numpy(), index=rt["doc_id"])
    comparacion_parametros[f"Posición: {nombre}"] = posiciones
    comparacion_parametros[f"Score: {nombre}"] = scores
    for doc in docs_elegidos:
        for termino in aportes.columns:
            filas_aportes.append({
                "Configuración": nombre, "k1": k1, "b": b,
                "doc_id": doc, "Término": termino,
                "f(t,d)": conteos[documentos.index(doc)].get(termino, 0),
                "Aporte BM25": aportes.loc[doc, termino],
            })
print("Consulta común a las tres ejecuciones:", consulta_elegida)
display(comparacion_parametros.round(8))
display(pd.DataFrame(filas_aportes).pivot(
    index=["doc_id", "Término", "f(t,d)"], columns="Configuración", values="Aporte BM25",
).round(8))
np.testing.assert_array_equal(vectorizador.idf_, idf_antes)
assert df == df_antes and avgdl == float(longitudes_antes.mean())
np.testing.assert_array_equal(longitudes, longitudes_antes)


Consulta común a las tres ejecuciones: love marriage family


,|d|,|d|/avgdl,Posición: Base,Score: Base,Posición: Sin ajuste por longitud,Score: Sin ajuste por longitud,Posición: Mayor efecto de las repeticiones,Score: Mayor efecto de las repeticiones
doc_id,,,,,,,,
austen-emma.txt,160994,1.367616,3.0,2.061886,2.0,2.074308,3.0,2.760910
austen-persuasion.txt,83616,0.710303,2.0,2.071438,3.0,2.058177,2.0,2.781712
austen-sense.txt,119912,1.018632,1.0,2.078310,1.0,2.078885,1.0,2.796706
bible-kjv.txt,790052,6.711354,4.0,1.844618,4.0,2.057215,4.0,2.341999
blake-poems.txt,6837,0.058079,12.0,1.123739,13.0,0.848830,12.0,1.390130
bryant-stories.txt,46319,0.393472,7.0,1.670943,8.0,1.470598,7.0,2.051586
burgess-busterbrown.txt,15963,0.135603,17.0,0.314982,17.0,0.203599,17.0,0.358574
carroll-alice.txt,26687,0.226701,14.0,0.883377,15.0,0.661691,15.0,1.034363
chesterton-ball.txt,82172,0.698037,13.0,1.066007,12.0,1.033732,13.0,1.358186


Configuración                         Base  Mayor efecto de las repeticiones  \
doc_id           Término  f(t,d)                                               
austen-sense.txt family   83      0.740986                          1.000795   
                 love     77      0.440950                          0.595123   
                 marriage 44      0.896375                          1.200787   
blake-poems.txt  family   0       0.000000                          0.000000   
                 love     29      0.442542                          0.598676   
                 marriage 1       0.681197                          0.791454   

Configuración                     Sin ajuste por longitud  
doc_id           Término  f(t,d)                           
austen-sense.txt family   83                     0.741133  
                 love     77                     0.441044  
                 marriage 44                     0.896707  
blake-poems.txt  family   0                      0.000000  
                 love     29                     0.430120  
                 marriage 1                      0.418710

**Lectura de los parámetros.** Las tres configuraciones mantienen fijos los documentos, tokens, frecuencias, DF, IDF y avgdl. Con $b=0$, el factor de longitud desaparece. A igual frecuencia, esto beneficia frente a la base a los documentos más largos que la media y reduce el aporte de los más cortos. Al pasar de $k_1=1.2$ a $2$, la frecuencia se satura más lentamente: con longitud media, el límite del factor pasa de 2.2 a 3. No equivale a multiplicar todos los scores por una constante.

Las posiciones se calculan sobre todos los documentos con score positivo, para seguir los mismos IDs aunque salgan del top-10. La tabla por término permite observar cambios de aporte incluso cuando la posición no cambia. Los parámetros son los pedidos en el enunciado, no una selección óptima: eso requeriría un conjunto de desarrollo y juicios de relevancia.

**Resultado en los dos documentos analizados.** Al quitar el ajuste de longitud, Blake baja del puesto 12 al 13 y su score pasa de 1.123739 a 0.848830. El aporte de `marriage`, con una sola aparición, cae de 0.681197 a 0.418710: pierde la compensación por ser mucho más corto que la media. Austen, ligeramente más largo que la media, aumenta de 2.078310 a 2.078885 y conserva el puesto 1.

Con $k_1=2$, Blake sube a 1.390130 y Austen a 2.796706, pero mantienen los puestos 12 y 1. Las contribuciones cambian, como muestra la tabla por término; el cambio no supera a los documentos que los separan en la ordenación ni revierte la ventaja de Austen. La presencia de los tres términos y sus frecuencias altas siguen dando a Austen un score mayor. Cambiar parámetros puede modificar scores sin modificar posiciones.